# LightGBM Nested Rolling-Origin Experiment

This notebook inspects and launches the fixed contract-v2 experiment. It does not redefine folds, feature order, sampling, or holdout timestamps. The command-line trainer is the single implementation used for reportable results. SHAP is intentionally out of scope.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROTOCOL_PATH = PROJECT_ROOT / 'configs' / 'forecast_experiment_protocol.json'
MANIFEST_PATH = PROJECT_ROOT / 'configs' / 'forecast_fold_manifest_v2.json'
SNAPSHOT_PATH = PROJECT_ROOT / 'artifacts' / 'manifests' / 'training_snapshot.json'
MODEL_ROOT = PROJECT_ROOT / 'models' / 'lightgbm'
EVALUATION_ROOT = PROJECT_ROOT / 'data' / 'gold' / 'lightgbm_nested_evaluation'
print('Project root:', PROJECT_ROOT)

## 1. Inspect The Frozen Protocol

In [ ]:
protocol = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['manifest_version'] == 2
assert manifest['manifest_id'] == protocol['manifest_id']
print(json.dumps(protocol, indent=2))
print('Outer blocks:', len(manifest['outer_blocks']))
print('Final holdout:', manifest['final_holdout'])

## 2. Display Rolling Windows

Each outer row is a non-overlapping UTC-day evaluation. Objective selection happens only in that row's nested inner blocks.

In [ ]:
import pandas as pd

outer_rows = []
for block in manifest['outer_blocks']:
    outer_rows.append({
        'block_id': block['block_id'],
        'training_start': block['training_start_hour'],
        'training_end_exclusive': block['training_end_hour_exclusive'],
        'evaluation_start': block['evaluation_start_hour'],
        'evaluation_end_exclusive': block['evaluation_end_hour_exclusive'],
        'inner_blocks': len(block['inner_blocks']),
    })
pd.DataFrame(outer_rows)

## 3. Check Data And Snapshot Readiness

In [ ]:
features_dir = PROJECT_ROOT / manifest['dataset']
parquet_files = list(features_dir.rglob('*.parquet')) if features_dir.exists() else []
print('Forecast feature files:', len(parquet_files))
print('Training snapshot exists:', SNAPSHOT_PATH.is_file())
if SNAPSHOT_PATH.is_file():
    snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding='utf-8'))
    print(json.dumps({key: snapshot.get(key) for key in [
        'snapshot_id', 'dataset', 'row_count', 'dataset_start_hour',
        'dataset_end_hour', 'git_commit', 'git_dirty'
    ]}, indent=2))
else:
    print('Build this only after the complete multi-week Gold snapshot is validated.')

## 4. Build The Training Snapshot

Run only after source is committed and contract-v2 Gold covers the manifest. The manifest hashes every feature Parquet file and contract source.

In [ ]:
build_snapshot_command = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'build_snapshot_manifest.py')
]
print(' '.join(build_snapshot_command))
# subprocess.run(build_snapshot_command, cwd=PROJECT_ROOT, check=True)

## 5. Train And Evaluate

The trainer verifies every snapshot hash, selects Poisson/Tweedie/L1 inside nested rolling windows, scores outer UTC-day blocks, opens the seven-day holdout once, and publishes a version only after reports succeed.

In [ ]:
train_command = [
    sys.executable, str(PROJECT_ROOT / 'spark_jobs' / 'train_lightgbm_nested.py')
]
print(' '.join(train_command))
# subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)

## 6. Inspect The Published Version

In [ ]:
current_path = MODEL_ROOT / 'current.json'
if not current_path.is_file():
    print('No contract-v2 model has been published.')
else:
    current = json.loads(current_path.read_text(encoding='utf-8'))
    version = current['model_version']
    version_dir = MODEL_ROOT / current['path']
    metadata = json.loads((version_dir / 'metadata.json').read_text(encoding='utf-8'))
    print(json.dumps(metadata, indent=2))
    print('Artifact files:', [path.name for path in version_dir.iterdir()])
    evaluation_dir = EVALUATION_ROOT / version
    print('Evaluation files:', [path.name for path in evaluation_dir.iterdir()])

## 7. Compare Paired Traffic And Ranking Results

In [ ]:
if current_path.is_file():
    outer = pd.read_parquet(evaluation_dir / 'outer_metrics.parquet')
    ranking = pd.read_parquet(evaluation_dir / 'outer_ranking_metrics.parquet')
    confidence = json.loads((evaluation_dir / 'block_bootstrap_confidence_intervals.json').read_text(encoding='utf-8'))
    display(outer.sort_values(['block_id', 'forecast_method']))
    display(ranking.sort_values(['block_id', 'k', 'forecast_method']).head(50))
    print(json.dumps(confidence, indent=2))